# Topical Lectures, Controls
## Flight data, part 2: the figure for the summary

Andreas Freise, 01.11.2026

This notebook needs no drone. It can be used to produce figures and the table of your group for a summary. You need the flight files, a copy of the `data/` folder of the drone laptop, taken once the pilot has flown what you want to show. Any model flights saved into that folder will be used too. Run the notebook again when new flights arrive. 

In [ ]:
# Importing packages
import numpy as np
import matplotlib.pyplot as plt
import module
import crazyflie

%matplotlib inline

## Loading

`crazyflie.load_runs` loads every saved run of the folder, oldest first. Each run holds the loop records (`pid`: time, position, velocity, target and feedback per iteration), the axes the controller acted on, the loop time of the step, and the three parts of its name. Runs aborted before the step are left out below.


In [ ]:
DATA = 'data'
runs = crazyflie.load_runs(DATA)
if not runs:
    DATA = 'data/reference'
    runs = crazyflie.load_runs(DATA)
    print('no flights in data/ yet, using the recorded reference flights')
runs = [r for r in runs if r['t_step'] is not None]
for r in runs:
    print(f"{r['label']:30s}  drone {str(r['drone']):6s}  axes {r['axes']:2s}  {len(r['pid']['t_s'])} loop samples")


## The numbers side by side

`crazyflie.run_table` gives, per run and axis the controller acted on, the gains the run flew with, when they were saved with it, and the numbers of its step: the rise time to 90 % of the step, the overshoot, the largest and the mean error in the last 2 s and the fraction of the step in which the feedback sat at the limit. One line per run and axis.


In [ ]:
rows = crazyflie.run_table(runs)

## Step responses on top of each other

Time runs from the step, the position is counted from the target before the step. Model flights are dashed, flights under the firmware dotted. `SELECT` picks runs by name, `None` takes all.


In [ ]:
SELECT = None     # e.g. ['A_1541_kp05', 'model_1312_kp05'], None for all

chosen = [r for r in runs if SELECT is None or r['label'] in SELECT]
fig, ax = plt.subplots(figsize=(10, 5))
for r in chosen:
    pid = r['pid']
    t = pid['t_s'] - r['t_step']
    for a in r['axes']:
        name = r['label'] if len(r['axes']) == 1 else f"{r['label']} {a}"
        if 'Kp' in r['info']:
            name += f"  Kp {r['info']['Kp']:+.1f}, Kd {r['info']['Kd']:+.1f}, Ki {r['info']['Ki']:+.1f}"
        style = '--' if r['drone'] == 'model' else (':' if (r['run_label'] or '').startswith('firmware') else '-')
        ax.plot(t, pid[a] - pid[f'target_{a}'][0], ls=style, label=name)
ax.axhspan(0.5 - 0.15, 0.5 + 0.15, color='0.92', zorder=0)
ax.axhline(0.5, color='k', lw=0.5, ls='--')
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_xlim(-1, 8)
ax.set_xlabel('time since step [s]')
ax.set_ylabel('position from start [m]')
ax.legend(loc='lower right', fontsize=8)
ax.grid()


## Paths in the plane

For runs on both axes the step goes diagonally. The path shows whether the two axes behave the same; a bend means one axis is faster than the other.


In [ ]:
both = [r for r in chosen if r['axes'] == 'xy']
if both:
    fig, ax = plt.subplots(figsize=(6, 6))
    for r in both:
        pid = r['pid']
        ax.plot(pid['x'] - pid['target_x'][0], pid['y'] - pid['target_y'][0], label=r['label'])
    ax.plot([0, 0.5], [0, 0.5], 'k+', ms=12, mew=2)
    ax.set_xlabel('x from start [m]')
    ax.set_ylabel('y from start [m]')
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
    ax.grid()
else:
    print('no run on both axes yet')


## The hold against the noise

In the hold phase before the step our controller holds the estimated position. Its spread cannot be smaller than the noise of the estimate, which the hover of the identification flight shows, before any loop of ours was closed; what is on top of it is the loop and the air. The cell prints the spread of every run next to that hover.


In [ ]:
ident = [x for x in runs if (x['run_label'] or '').startswith('identify')]
if ident:
    p = ident[-1]['pid']; hover = p['step'] < 0.5
    print(f"hover of the identification flight: std x {1000 * np.std(p['x'][hover]):5.1f} mm, y {1000 * np.std(p['y'][hover]):5.1f} mm")
else:
    print('no identification flight in the folder')
for x in chosen:
    pid = x['pid']
    hold = pid['step'] < 0.5
    for a in x['axes']:
        print(f"{x['label']:30s} hold, axis {a}: std {1000 * np.std(pid[a][hold]):5.1f} mm, "
              f"wander {1000 * np.ptp(pid[a][hold]):5.1f} mm over {hold.sum() * 0.02:.1f} s")


## Our loop against the firmware

The firmware has a position loop of its own, tuned by the people who built the drone, and it solves the same task in a different way: our loop gets the new target at once and reacts to the error, the firmware moves its setpoint at a constant speed and follows it. The first flight of the afternoon under the firmware, 0.5 m forward and back, is in the folder with the tuning flights, so the two can be drawn over each other without another flight. Time runs from the step of our loop and from the start of the firmware's move; the dotted line is the setpoint the firmware followed. The firmware held for 2 s after its move, our loop for 8 s, so the arrival and the overshoot compare directly and the hold only over those two seconds. `OURS` picks the tuning flight, `None` takes the last one.

Which one arrives first, which one overshoots, which one holds better? Where does each approach spend its effort, and what would our loop need to do the same? And where would you put the intelligence in a system you build yourself: into the loop, or into the setpoint you give it?


In [ ]:
OURS = None       # the label of a tuning flight, None for the last real flight with gains
firmware = [x for x in runs if (x['run_label'] or '').startswith('firmware')]
tuned = [x for x in runs if x['drone'] != 'model' and 'Kp' in x['info']]
if not firmware or not tuned:
    print('needs one flight under the firmware and one tuning flight in the folder')
else:
    fw = firmware[0]
    ours = tuned[-1] if OURS is None else next(x for x in tuned if x['label'] == OURS)
    po, pf = ours['pid'], fw['pid']
    to, tf = po['t_s'] - ours['t_step'], pf['t_s'] - fw['t_step']
    g = ours['info']
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(to, po['x'] - po['target_x'][0], label=f"our loop, Kp = {g['Kp']:+.2f}, Kd = {g['Kd']:+.2f}, Ki = {g['Ki']:+.2f}")
    ax.plot(tf, pf['x'] - pf['x'][0], label='firmware')
    ax.plot(tf, pf['target_x'] - pf['target_x'][0], 'k:', lw=0.8, label='setpoint of the firmware')
    ax.axhspan(0.5 - 0.15, 0.5 + 0.15, color='0.92', zorder=0)
    ax.axhline(0.5, color='k', lw=0.5, ls='--')
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.set_xlim(-1, 8)
    ax.set_xlabel('time since the step, or the start of the move [s]')
    ax.set_ylabel('position from start [m]')
    ax.legend(loc='lower right')
    ax.grid()
    rows = crazyflie.run_table([ours, fw])


## The figure for the summary

One figure per group for the summary: the flights of your drone that tell the story of the afternoon, with the model for the gains you settled on drawn over them, and a legend that names what changed. Put the run names into `SUMMARY`. The model is the one of the model drone notebook, written out here once more, fed with the gains and the parameters saved with the last run shown.


In [ ]:
def model_flight(Kp, Kd, Ki, tau, delay, step=0.5, v_max=0.3, dt=0.02, hold_s=2.0, step_s=4.0):
    """One step of the target on the model drone. Returns t (seconds from the step), x, v and the feedback."""
    n, n_delay = int(round((hold_s + step_s) / dt)), int(round(delay / dt))
    t = np.arange(n) * dt - hold_s
    target = np.where(t >= 0, step, 0.0)
    x, v, fb = np.zeros(n), np.zeros(n), np.zeros(n)
    ctrl = module.PIDController(Kp=Kp, Ki=Ki, Kd=Kd, deltaT=dt)
    for i in range(1, n):
        fb[i - 1] = float(np.clip(ctrl(x[i - 1], v[i - 1], target[i - 1]), -v_max, v_max))   # what our loop sends
        acting = fb[i - 1 - n_delay] if i - 1 - n_delay >= 0 else 0.0                        # what reaches the drone now
        v[i] = v[i - 1] + (acting - v[i - 1]) * dt / tau                                      # the velocity loop follows
        x[i] = x[i - 1] + v[i] * dt                                                           # the position is its integral
    return t, x, v, fb

SUMMARY = [x['label'] for x in runs if x['drone'] != 'model'][-3:]   # the last three real flights; edit

shown = [x for x in runs if x['label'] in SUMMARY]
fig, ax = plt.subplots(figsize=(10, 5))
for x in shown:
    pid = x['pid']
    t = pid['t_s'] - x['t_step']
    a = x['axes'][0]
    ax.plot(t, pid[a] - pid[f'target_{a}'][0], label=x['run_label'] or x['label'])
final = shown[-1]['info'] if shown else {}
if all(k in final for k in ('Kp', 'Kd', 'Ki', 'tau', 'delay')):
    t_m, x_m, _, _ = model_flight(final['Kp'], final['Kd'], final['Ki'], final['tau'], final['delay'], step=final.get('step', 0.5), v_max=final.get('v_max', 0.3))
    ax.plot(t_m, x_m, 'k--', label=f"model, Kp = {final['Kp']:+.2f}, Kd = {final['Kd']:+.2f}, Ki = {final['Ki']:+.2f}")
else:
    print('the last run shown carries no gains and parameters; no model curve')
ax.axhspan(0.5 - 0.15, 0.5 + 0.15, color='0.92', zorder=0)
ax.axhline(0.5, color='k', lw=0.5, ls='--')
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_xlim(-1, 8)
ax.set_xlabel('time since step [s]')
ax.set_ylabel('position from start [m]')
ax.legend(loc='lower right')
ax.grid()
if shown:
    out = f"data/summary_{shown[0]['drone']}_{shown[-1]['when']}.png"
    fig.savefig(out, dpi=150, bbox_inches='tight')
    print('figure saved as', out)
else:
    print('no real flights in the folder yet')
